In [1]:
# Install dependencies if they are not already installed
%pip install -q tensorflow tensorflow-datasets matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

print("TensorFlow:", tf.__version__)
print("TensorFlow Datasets:", tfds.__version__)

TensorFlow: 2.20.0
TensorFlow Datasets: 4.9.10


In [ ]:
(train_raw, val_raw, test_raw), info = tfds.load(
    "voc/2007",
    split=["train", "validation", "test"],
    with_info=True
)

class_names = info.features["objects"]["label"].names
num_classes = len(class_names)

print("Dataset: PASCAL VOC 2007")
print("Number of classes:", num_classes)
print("Classes:", class_names)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def preprocess_sample(sample):
    image = tf.image.resize(sample["image"], (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)

    # VOC can contain multiple objects in one image.
    # Convert all object labels into a 20-element multi-hot vector.
    object_labels = tf.cast(sample["objects"]["label"], tf.int32)
    one_hot_labels = tf.one_hot(object_labels, depth=num_classes)
    multi_hot_labels = tf.reduce_max(one_hot_labels, axis=0)

    image = preprocess_input(image)
    return image, multi_hot_labels

train_ds = (
    train_raw
    .map(preprocess_sample, num_parallel_calls=AUTOTUNE)
    .shuffle(1000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_raw
    .map(preprocess_sample, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    test_raw
    .map(preprocess_sample, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Training, validation and test datasets are ready.")

In [ ]:
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False
print("VGG16 loaded. Trainable base layers:",
      sum(layer.trainable for layer in base_model.layers))

In [ ]:
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)

# Sigmoid gives an independent probability for each of the 20 classes.
outputs = layers.Dense(num_classes, activation="sigmoid")(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="binary_accuracy", threshold=0.5),
        tf.keras.metrics.AUC(name="auc", multi_label=True, num_labels=num_classes)
    ]
)

model.summary()

In [ ]:
INITIAL_EPOCHS = 5

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    verbose=1
)

In [ ]:
base_model.trainable = True

for layer in base_model.layers:
    layer.trainable = layer.name.startswith("block5_")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="binary_accuracy", threshold=0.5),
        tf.keras.metrics.AUC(name="auc", multi_label=True, num_labels=num_classes)
    ]
)

FINE_TUNE_EPOCHS = 3

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    verbose=1
)

In [ ]:
test_results = model.evaluate(test_ds, return_dict=True, verbose=1)

for metric_name, metric_value in test_results.items():
    print(f"{metric_name}: {metric_value:.4f}")

In [ ]:
train_acc = history.history["binary_accuracy"] + history_ft.history["binary_accuracy"]
val_acc = history.history["val_binary_accuracy"] + history_ft.history["val_binary_accuracy"]
train_loss = history.history["loss"] + history_ft.history["loss"]
val_loss = history.history["val_loss"] + history_ft.history["val_loss"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(train_acc, label="Training binary accuracy")
axes[0].plot(val_acc, label="Validation binary accuracy")
axes[0].axvline(INITIAL_EPOCHS - 1, linestyle="--", label="Fine-tuning starts")
axes[0].set_title("Binary Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(train_loss, label="Training loss")
axes[1].plot(val_loss, label="Validation loss")
axes[1].axvline(INITIAL_EPOCHS - 1, linestyle="--", label="Fine-tuning starts")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
for image_batch, label_batch in test_ds.take(1):
    probabilities = model.predict(image_batch[:1], verbose=0)[0]
    actual_vector = label_batch[0].numpy()

    actual_ids = np.where(actual_vector >= 0.5)[0]
    predicted_ids = np.where(probabilities >= 0.5)[0]

    # If no class crosses 0.5, show the three highest-scoring classes.
    if len(predicted_ids) == 0:
        predicted_ids = np.argsort(probabilities)[-3:][::-1]

    # VGG16 preprocessing changes pixel values; reverse it for display.
    display_image = image_batch[0].numpy().copy()
    display_image = display_image[..., ::-1]  # BGR to RGB
    display_image += [103.939, 116.779, 123.68]
    display_image = np.clip(display_image, 0, 255).astype("uint8")

    plt.figure(figsize=(7, 7))
    plt.imshow(display_image)
    plt.axis("off")
    plt.title("PASCAL VOC 2007 sample")
    plt.show()

    print("Actual labels:",
          [class_names[i] for i in actual_ids])
    print("Predicted labels:",
          [(class_names[i], round(float(probabilities[i]), 3))
           for i in predicted_ids])